In [ ]:
import numpy as np
import pandas as pd


df = pd.read_csv("dummydata.csv")
print(df)

# 2) Simple custom train-test split 
def custom_train_test_split(X, y, test_size=0.2, random_state=42):
    np.random.seed(random_state)
    n = len(X)
    indices = np.random.permutation(n)

    test_n = int(np.ceil(n * test_size))
    test_idx = indices[:test_n]
    train_idx = indices[test_n:]

    X_train = X[train_idx]
    X_test = X[test_idx]
    y_train = y[train_idx]
    y_test = y[test_idx]

    return X_train, X_test, y_train, y_test

# 3) simple linear regression: y = m*x + b
def fit_simple_linear_regression(x, y):
    x_mean = np.mean(x)
    y_mean = np.mean(y)

    # m = covariance(x, y) / variance(y)
    cov_xy = np.mean((x - x_mean) * (y - y_mean))
    var_y = np.mean((y - y_mean) ** 2)
    m = cov_xy / var_y
    b = y_mean - m * x_mean
    return m, b

def predict(x, m, b):
    return m * x + b

# 4) Use one feature for simple regression
x = df["study_hours"].to_numpy(dtype=float)
y = df["marks"].to_numpy(dtype=float)

X_train, X_test, y_train, y_test = custom_train_test_split(x, y, test_size=0.2, random_state=42)

m, b = fit_simple_linear_regression(X_train, y_train)
y_pred = predict(X_test, m, b)

print("\nTrain size:", len(X_train), "Test size:", len(X_test))
print("m:", m)
print("b:", b)
print("\nX_test:", X_test)
print("y_test:", y_test)
print("y_pred:", y_pred)

   study_hours  leave_count  other  marks
0            4            1      3     52
1            9            2      1     51
2            8            3      2     50
3            8            1      1     52
4            7            1      4     70
5            5            0      2     60

Train size: 4 Test size: 2
m: -0.07258064516129033
b: 58.50806451612903

X_test: [4. 9.]
y_test: [52. 51.]
y_pred: [58.21774194 57.85483871]


In [5]:
# 5) Multiple linear regression (target: marks) using all remaining columns as features
feature_cols = ["study_hours", "leave_count", "other"]
target_col = "marks"

X_multi = df[feature_cols].to_numpy(dtype=float)
y_multi = df[target_col].to_numpy(dtype=float)

X_train_m, X_test_m, y_train_m, y_test_m = custom_train_test_split(
    X_multi, y_multi, test_size=0.33, random_state=42
)

def fit_multiple_linear_regression(X, y):
    # Add bias/intercept column
    X_b = np.c_[np.ones((X.shape[0], 1)), X]
    # Normal equation with pseudo-inverse for numerical stability
    theta = np.linalg.pinv(X_b.T @ X_b) @ X_b.T @ y
    return theta

def predict_multiple_linear_regression(X, theta):
    X_b = np.c_[np.ones((X.shape[0], 1)), X]
    return X_b @ theta

def mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

theta = fit_multiple_linear_regression(X_train_m, y_train_m)
y_pred_m = predict_multiple_linear_regression(X_test_m, theta)

print("\nFeatures:", feature_cols)
print("Train size:", len(X_train_m), "Test size:", len(X_test_m))
print("Coefficients (intercept first):", theta)
print("\nX_test:\n", X_test_m)
print("y_test:", y_test_m)
print("y_pred:", np.round(y_pred_m, 2))
print("MSE:", round(mse(y_test_m, y_pred_m), 4))


Features: ['study_hours', 'leave_count', 'other']
Train size: 4 Test size: 2
Coefficients (intercept first): [43.46666667  0.8        -4.13333333  6.26666667]

X_test:
 [[4. 1. 3.]
 [9. 2. 1.]]
y_test: [52. 51.]
y_pred: [61.33 48.67]
MSE: 46.2778


In [ ]:
# 6) Very basic logistic probability (no training, no epochs)
# We assume coefficients directly and compute p using sigmoid.

import numpy as np

def sigmoid(z):
    return 1 / (1 + np.exp(-z))


x = np.array([6, 1, 1], dtype=float)

# Assumed coefficients
b0 = -3.0   # intercept
b1 = 0.8    # study_hours weight
b2 = -0.5   # leave_count weight
b3 = 0.4    # other weight

# Linear score z = b0 + b1*x1 + b2*x2 + b3*x3
z = b0 + b1 * x[0] + b2 * x[1] + b3 * x[2]

# Probability p = sigmoid(z)
p = sigmoid(z)

print("Input [study_hours, leave_count, other]:", x.tolist())
print("z:", round(float(z), 4))
print("p (probability of class 1):", round(float(p), 4))
print("Predicted class:", 1 if p >= 0.5 else 0)

In [ ]:
# Same idea: z = X_b @ theta, but output is sigmoid(z) so prediction is a probability.

X_log = np.array([
    [1, 5, 0],
    [2, 4, 1],
    [3, 3, 0],
    [4, 2, 1],
    [5, 2, 1],
    [6, 1, 0],
    [7, 1, 1],
    [8, 0, 1]
], dtype=float)

y_log = np.array([0, 0, 0, 0, 1, 1, 1, 1], dtype=float)

# Reuse split function from above
X_train_l, X_test_l, y_train_l, y_test_l = custom_train_test_split(X_log, y_log, test_size=0.25, random_state=42)

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def fit_logistic_like_linear(X, y, lr=0.1, steps=1500):
    X_b = np.c_[np.ones((X.shape[0], 1)), X]  # same bias-column style as linear regression
    theta = np.zeros(X_b.shape[1], dtype=float)

    for _ in range(steps):
        p = sigmoid(X_b @ theta)
        grad = (X_b.T @ (p - y)) / len(y)
        theta -= lr * grad

    return theta

def predict_logistic_probability(X, theta):
    X_b = np.c_[np.ones((X.shape[0], 1)), X]
    return sigmoid(X_b @ theta)

theta_log2 = fit_logistic_like_linear(X_train_l, y_train_l, lr=0.1, steps=1500)
p_test = predict_logistic_probability(X_test_l, theta_log2)
y_pred_cls = (p_test >= 0.5).astype(int)

print('Train size:', len(X_train_l), 'Test size:', len(X_test_l))
print('Theta (intercept first):', np.round(theta_log2, 4))
print('Probabilities p:', np.round(p_test, 4))
print('Predicted class:', y_pred_cls)
print('Actual class:', y_test_l.astype(int))